In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import math

C:\Users\Apoorv Bagga\AppData\Roaming\Python\Python310\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# ─────────────────────────────────────────────
# 1. DATASET  (same as Component I)
# ─────────────────────────────────────────────
corpus = """
machine learning models learn patterns from data.
sequence models process data step by step.
recurrent neural networks are designed for sequential tasks.
rnn models maintain hidden states across time steps.
long short term memory networks solve long dependency problems.
lstm uses gates to control information flow.
gru models simplify the lstm architecture.
sequence prediction is useful in many applications.
language modeling predicts the next word in a sentence.
speech recognition processes audio sequences.
time series forecasting predicts future values.
music generation creates new melodies.
generative models learn probability distributions.
they generate new samples similar to training data.
sequence generation is widely used in artificial intelligence.
deep learning improves sequence modeling performance.
""".strip()

In [3]:
# ─────────────────────────────────────────────
# 2. WORD-LEVEL TOKENIZATION & VOCABULARY
# ─────────────────────────────────────────────
words      = corpus.lower().split()
vocab      = sorted(set(words))

# Reserve two special tokens
PAD_TOKEN  = "<PAD>"
UNK_TOKEN  = "<UNK>"
special    = [PAD_TOKEN, UNK_TOKEN]
vocab      = special + vocab

word2idx   = {w: i for i, w in enumerate(vocab)}
idx2word   = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)
PAD_IDX    = word2idx[PAD_TOKEN]

print(f"Vocabulary size : {vocab_size}")
print(f"Total words     : {len(words)}\n")

Vocabulary size : 90
Total words     : 111



In [4]:
# ─────────────────────────────────────────────
# 3. BUILD INPUT-OUTPUT PAIRS
# ─────────────────────────────────────────────
SEQ_LEN = 5

def build_sequences(words, seq_len):
    indices = [word2idx.get(w, word2idx[UNK_TOKEN]) for w in words]
    X, y   = [], []
    for i in range(len(indices) - seq_len):
        X.append(indices[i : i + seq_len])
        y.append(indices[i + seq_len])
    return np.array(X), np.array(y)

X, y        = build_sequences(words, SEQ_LEN)
X_tensor    = torch.tensor(X, dtype=torch.long)
y_tensor    = torch.tensor(y, dtype=torch.long)

print(f"Training samples: {len(X)}\n")


Training samples: 106



In [5]:
# ─────────────────────────────────────────────
# 4. POSITIONAL ENCODING
# ─────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    """
    Injects position information into token embeddings using
    sine and cosine functions of different frequencies.
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe  = torch.zeros(max_len, d_model)          # (max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()   # (max_len, 1)
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)           # even indices
        pe[:, 1::2] = torch.cos(pos * div)           # odd indices
        pe = pe.unsqueeze(0)                         # (1, max_len, d_model)
        self.register_buffer("pe", pe)               # not a trainable param

    def forward(self, x):
        # x : (batch, seq_len, d_model)
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


In [6]:
# ─────────────────────────────────────────────
# 5. TRANSFORMER MODEL DEFINITION
# ─────────────────────────────────────────────
class TransformerTextGenerator(nn.Module):
    """
    Decoder-style Transformer for next-word prediction.
    Architecture:
        Embedding → Positional Encoding
        → N × TransformerEncoderLayer (with causal mask)
        → Linear projection → vocab logits
    """
    def __init__(self, vocab_size, d_model, nhead,
                 num_layers, dim_ff, dropout=0.1):
        super(TransformerTextGenerator, self).__init__()

        self.embedding  = nn.Embedding(vocab_size, d_model,
                                       padding_idx=PAD_IDX)
        self.pos_enc    = PositionalEncoding(d_model, dropout=dropout)

        encoder_layer   = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer,
                                                 num_layers=num_layers)
        self.fc_out      = nn.Linear(d_model, vocab_size)
        self.d_model     = d_model

    @staticmethod
    def _causal_mask(seq_len):
        """Upper-triangular mask so each position only attends to past tokens."""
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        return mask  # True = ignore

    def forward(self, x):
        # x : (batch, seq_len)
        seq_len  = x.size(1)
        emb      = self.embedding(x) * math.sqrt(self.d_model)
        emb      = self.pos_enc(emb)

        causal_m = self._causal_mask(seq_len).to(x.device)
        out      = self.transformer(emb, mask=causal_m)
        logits   = self.fc_out(out[:, -1, :])   # use last position
        return logits

In [7]:
# ─────────────────────────────────────────────
# Hyper-parameters
# ─────────────────────────────────────────────
D_MODEL    = 64
NHEAD      = 4     # must divide D_MODEL evenly
NUM_LAYERS = 2
DIM_FF     = 128
DROPOUT    = 0.1

model     = TransformerTextGenerator(vocab_size, D_MODEL, NHEAD,
                                     NUM_LAYERS, DIM_FF, DROPOUT)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=0.005)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.7)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}\n")

TransformerTextGenerator(
  (embedding): Embedding(90, 64, padding_idx=0)
  (pos_enc): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc_out): Linear(in_features=64, out_features=90, bias=True)
)

Trainable parameters: 78,554



In [8]:
# ─────────────────────────────────────────────
# 6. TRAINING
# ─────────────────────────────────────────────
EPOCHS     = 200
BATCH_SIZE = 32

dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
loader  = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE,
                                       shuffle=True)

model.train()
for epoch in range(1, EPOCHS + 1):
    total_loss = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()

    if epoch % 40 == 0:
        avg_loss = total_loss / len(loader)
        lr_now   = optimizer.param_groups[0]["lr"]
        print(f"Epoch [{epoch:3d}/{EPOCHS}]  Loss: {avg_loss:.4f}  LR: {lr_now:.5f}")

print("\nTraining complete!\n")

# ─────────────────────────────────────────────
# 7. SEQUENCE GENERATION
# ─────────────────────────────────────────────
def generate_sequence(model, seed_words, num_words=10, temperature=0.8):
    """
    Autoregressively generate words one at a time from a seed.
    temperature < 1  → sharper distribution (more confident)
    temperature > 1  → flatter distribution (more diverse)
    """
    model.eval()
    result   = list(seed_words)
    seed_idx = [word2idx.get(w, word2idx[UNK_TOKEN]) for w in seed_words]

    with torch.no_grad():
        for _ in range(num_words):
            context = seed_idx[-SEQ_LEN:]
            inp     = torch.tensor([context], dtype=torch.long)
            logits  = model(inp)
            probs   = torch.softmax(logits / temperature, dim=-1).squeeze()
            # exclude special tokens from sampling
            probs[PAD_IDX]                      = 0
            probs[word2idx[UNK_TOKEN]]          = 0
            probs = probs / probs.sum()
            next_idx = torch.multinomial(probs, 1).item()
            result.append(idx2word[next_idx])
            seed_idx.append(next_idx)

    return " ".join(result)


print("=" * 65)
print("GENERATED SEQUENCES (TRANSFORMER)")
print("=" * 65)

seeds = [
    ["machine",    "learning",  "models",  "learn",    "patterns"],
    ["long",       "short",     "term",    "memory",   "networks"],
    ["sequence",   "models",    "process", "data",     "step"],
    ["generative", "models",    "learn",   "probability", "distributions"],
    ["deep",       "learning",  "improves","sequence", "modeling"],
]

for seed in seeds:
    out = generate_sequence(model, seed, num_words=10, temperature=0.8)
    print(f"\nSeed   : {' '.join(seed)}")
    print(f"Output : {out}")

print("\n" + "=" * 65)


Epoch [ 40/200]  Loss: 0.0059  LR: 0.00500
Epoch [ 80/200]  Loss: 0.0017  LR: 0.00350
Epoch [120/200]  Loss: 0.0011  LR: 0.00245
Epoch [160/200]  Loss: 0.0008  LR: 0.00171
Epoch [200/200]  Loss: 0.0006  LR: 0.00120

Training complete!

GENERATED SEQUENCES (TRANSFORMER)

Seed   : machine learning models learn patterns
Output : machine learning models learn patterns from data. sequence models process data step by step. recurrent

Seed   : long short term memory networks
Output : long short term memory networks solve long dependency problems. lstm uses gates to control information

Seed   : sequence models process data step
Output : sequence models process data step by step. recurrent neural networks are designed for sequential tasks.

Seed   : generative models learn probability distributions
Output : generative models learn probability distributions a sentence. speech recognition processes audio sequences. time series forecasting

Seed   : deep learning improves sequence modeling
Output

In [9]:
# ─────────────────────────────────────────────
# 8. QUICK COMPARISON TABLE
# ─────────────────────────────────────────────
print("""
╔══════════════════╦══════════════════════╦══════════════════════╗
║ Aspect           ║ LSTM (Component I)   ║ Transformer (Comp II)║
╠══════════════════╬══════════════════════╬══════════════════════╣
║ Context handling ║ Hidden state (serial)║ Self-attention (para)║
║ Long-range deps  ║ Moderate             ║ Excellent            ║
║ Parallelism      ║ Low (sequential)     ║ High (all at once)   ║
║ Positional info  ║ Implicit via order   ║ Explicit (PE layer)  ║
║ Scalability      ║ Harder to scale      ║ Scales well          ║
╚══════════════════╩══════════════════════╩══════════════════════╝
""")



╔══════════════════╦══════════════════════╦══════════════════════╗
║ Aspect           ║ LSTM (Component I)   ║ Transformer (Comp II)║
╠══════════════════╬══════════════════════╬══════════════════════╣
║ Context handling ║ Hidden state (serial)║ Self-attention (para)║
║ Long-range deps  ║ Moderate             ║ Excellent            ║
║ Parallelism      ║ Low (sequential)     ║ High (all at once)   ║
║ Positional info  ║ Implicit via order   ║ Explicit (PE layer)  ║
║ Scalability      ║ Harder to scale      ║ Scales well          ║
╚══════════════════╩══════════════════════╩══════════════════════╝

